# B4 jeepney RL baseline

Traffic-biased quadrilateral baselines on the physical drivable street network.
The minimum-area rule is inspired by polygonal morphology bounds discussed in
https://arxiv.org/html/2603.28385v1.


## 1. Traffic-Biased Quadrilateral Baseline Generator

Builds traffic-weighted quadrilateral routes on the physical drive graph, then stitches the shortest physical path between the four selected anchors.

In [1]:
import pandas as pd
import secrets
from pathlib import Path
from types import SimpleNamespace

from IPython.display import IFrame, display
from tqdm.auto import tqdm

from _bootstrap import ROOT
from utils import BaselineRoute, BaselineRouteGenerator, JeepneySystem, JeepneyRouteEnv, SystemicFitnessEvaluator, calculate_route_fitness, train_route_agent

OUTPUT_DIR = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis") / "results" / "B4_baseline_routes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_ROUTES = 20
random_seed = secrets.randbits(32)
generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=random_seed,
)

routes = generator.generate_routes(NUM_ROUTES, route_prefix="B4", seed=random_seed)
route_system = JeepneySystem.from_routes(routes)

summary = pd.DataFrame(
    [
        {
            "route_id": route.route_id,
            "anchors": list(route.ordered_anchor_node_ids),
            "area_m2": round(route.polygon_area_m2, 0),
            "length_m": round(route.path_length_m, 0),
            "attempt": route.attempt_index,
        }
        for route in routes
    ]
)
summary


c:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,route_id,anchors,area_m2,length_m,attempt
0,B401,"[473, 1090, 1816, 3523]",33714810.0,36170.0,1
1,B402,"[2248, 2448, 1098, 1090]",90396184.0,80171.0,1
2,B403,"[2161, 21, 2448, 1240]",20858948.0,43480.0,1
3,B404,"[104, 745, 2812, 42]",15436689.0,39817.0,2
4,B405,"[2743, 1797, 89, 811]",57769893.0,72433.0,1
5,B406,"[2650, 2641, 873, 105]",105873508.0,58735.0,1
6,B407,"[2016, 528, 839, 1681]",18563780.0,42808.0,1
7,B408,"[605, 486, 768, 3238]",4417838.0,14097.0,1
8,B409,"[964, 856, 3241, 486]",113446108.0,71650.0,1
9,B410,"[809, 242, 869, 3203]",25078662.0,68400.0,1


## 2. RL Environment and Coordinate-Invariant Geometric Embeddings

Uses only the primal drivable network to build route geometry step by step, while encoding turn structure, sinuosity, and origin-relative features without absolute node IDs.

In [2]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
env = JeepneyRouteEnv(
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    seed=random_seed,
    max_steps=12,
)

def print_observation(obs: dict[str, np.ndarray]) -> None:
    print('shape:', obs['shape'])
    print('history:', obs['history'])
    print('topology:', obs['topology'])
    print('demand:', obs['demand'])
    print('global:', obs['global'])
    print('candidates:')
    for candidate_index, row in enumerate(obs['candidates']):
        print(f'  {candidate_index}: {row}')
    print('mask:', obs['action_mask'])

obs, info = env.reset()
print('reset state vector length:', info['state_vector'].shape[0])
print_observation(obs)

rng = np.random.default_rng(random_seed)
for step_index in range(5):
    valid_actions = np.flatnonzero(obs['action_mask'][:-1])
    action = int(rng.choice(valid_actions)) if len(valid_actions) else env.max_candidates
    obs, reward, terminated, truncated, info = env.step(action)
    print(f'step {step_index + 1}: action={action}, reward={reward:.3f}, terminated={terminated}, truncated={truncated}')
    print('turn angle rad:', info['turn_angle_rad'])
    print('sinuosity index:', info['sinuosity_index'])
    print('distance to origin m:', info['distance_to_origin_m'])
    print('bearing to origin rad:', info['bearing_to_origin_rad'])
    print('route area m2:', info['route_area_m2'])
    print('state vector:', info['state_vector'])
    print_observation(obs)
    if terminated or truncated:
        break


reset state vector length: 91
shape: [0. 0. 1. 0. 0. 0. 0.]
history: [0. 0. 0. 0. 0. 0. 0. 0.]
topology: [0.667 0.667 0.    0.5   0.556]
demand: [0.205 0.182 0.226 0.021]
global: [0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.667 0.667 0.    0.5   0.556 0.205 0.182 0.226 0.021]
candidates:
  0: [-0.978 -0.207  0.009  0.5    0.     0.226  0.021  0.     0.     0.   ]
  1: [ 0.022  1.     0.071  0.5    0.     0.186 -0.019  0.     0.     0.   ]
  2: [ 0.219 -0.976  0.025  0.667  0.     0.133 -0.072  0.     1.     0.   ]
  3: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  4: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  5: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
mask: [1. 1. 1. 0. 0. 0. 1.]
step 1: action=1, reward=-0.000, terminated=False, truncated=False
turn angle rad: 0.0
sinuosity index: 1.0
distance to origin m: 483.8593432889766
bearing to origin rad: 3.141592653589793
route area m2: 0.0
state vector: [ 0.007  0.    -1.     0.007  0.007  0.     0.     0.     0.     0.
  0.

In [3]:
route_edges = pd.DataFrame(
    [(int(u), int(v), "start_walk") for u, v in generator.drive_graph_raw.edges()],
    columns=["u", "v", "edge_type"],
)
route_manager = SimpleNamespace(
    edges=route_edges,
    _node_coords={
        int(row.base_node_id): (float(row.lat), float(row.lon))
        for row in generator.node_table.itertuples(index=False)
    },
)

def build_route_encoding(route_id: str) -> str:
    return f"route_id: {route_id}"

def interpret_embeddings(route: BaselineRoute) -> str:
    """Generate human-readable interpretations of route geometry."""
    route_path = route.path_node_ids
    if not route_path or len(route_path) < 2:
        return "Route too short to analyze."
    
    lines = []
    coords = route.path_latlon
    
    area = route.polygon_area_m2
    length = route.path_length_m
    area_to_length = area / max(length, 1.0)
    
    if area_to_length > 1000:
        lines.append("shape: Compact, efficient use of space")
    elif area_to_length > 200:
        lines.append("shape: Moderate coverage relative to distance")
    else:
        lines.append("shape: Linear route with extended length")
    
    node_count = len(route_path)
    if node_count > 50:
        lines.append("topology: High node density, complex route structure")
    elif node_count > 25:
        lines.append("topology: Moderate complexity with multiple waypoints")
    else:
        lines.append("topology: Simple, direct route path")
    
    if area > 50_000_000:
        lines.append("demand: Large service area, potentially high demand coverage")
    elif area > 10_000_000:
        lines.append("demand: Medium service area with moderate demand potential")
    else:
        lines.append("demand: Compact service area with focused demand")
    
    if len(coords) >= 2:
        import math
        start = coords[0]
        end = coords[-1]
        straight_dist = math.sqrt((end[0] - start[0])**2 + (end[1] - start[1])**2) * 111000
        actual_path = length
        sinuosity = actual_path / max(straight_dist, 1.0)
        
        if sinuosity > 1.5:
            lines.append("history: Winding route with many turns")
        elif sinuosity > 1.1:
            lines.append("history: Moderate turning pattern")
        else:
            lines.append("history: Direct path with minimal deviations")
    
    return chr(10).join(lines) if lines else "No embedding data available."

def format_route_fitness(result) -> str:
    return (
        f"standalone fitness: reward={result.reward:.3f} | avg_gtc={result.average_gtc:.3f} | "
        f"passenger_std={result.passenger_gtc_std:.3f}"
    )

def format_systemic_fitness(result) -> str:
    return (
        f"systemic fitness: avg_gtc={result.average_gtc:.3f} ± {result.std_gtc:.3f} | "
        f"avg_passenger_std={result.average_passenger_gtc_std:.3f} ± {result.std_passenger_gtc_std:.3f}"
    )

NOTE_SYSTEMIC_EVAL_TEST_MEAN = 3  # Lightweight note-time sample; increase later for final analysis.
NOTE_SYSTEMIC_EVAL_TEST_STD = 1
NOTE_SYSTEMIC_BACKGROUND_ROUTE_MEAN = 1
NOTE_SYSTEMIC_BACKGROUND_ROUTE_STD = 0
FULL_SYSTEMIC_EVAL_TEST_MEAN = 10  # Temporary placeholder; increase later for final sensitivity analysis.
FULL_SYSTEMIC_EVAL_TEST_STD = 5
FULL_SYSTEMIC_BACKGROUND_ROUTE_MEAN = 2
FULL_SYSTEMIC_BACKGROUND_ROUTE_STD = 1

note_systemic_evaluator = SystemicFitnessEvaluator(
    passenger_map=generator.passenger_map,
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    evaluation_test_mean=NOTE_SYSTEMIC_EVAL_TEST_MEAN,
    evaluation_test_std=NOTE_SYSTEMIC_EVAL_TEST_STD,
    background_route_mean=NOTE_SYSTEMIC_BACKGROUND_ROUTE_MEAN,
    background_route_std=NOTE_SYSTEMIC_BACKGROUND_ROUTE_STD,
    seed=random_seed,
)
full_systemic_evaluator = SystemicFitnessEvaluator(
    passenger_map=generator.passenger_map,
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    evaluation_test_mean=FULL_SYSTEMIC_EVAL_TEST_MEAN,
    evaluation_test_std=FULL_SYSTEMIC_EVAL_TEST_STD,
    background_route_mean=FULL_SYSTEMIC_BACKGROUND_ROUTE_MEAN,
    background_route_std=FULL_SYSTEMIC_BACKGROUND_ROUTE_STD,
    seed=random_seed,
)

standalone_fitness_cache: dict[tuple[int, ...], object] = {}
systemic_fitness_cache: dict[tuple[int, ...], object] = {}

def route_signature(route) -> tuple[int, ...]:
    return tuple(int(node_id) for node_id in route.path_node_ids)

def get_standalone_fitness(route):
    key = route_signature(route)
    cached = standalone_fitness_cache.get(key)
    if cached is None:
        cached = calculate_route_fitness(
            route.path_node_ids,
            passenger_map=generator.passenger_map,
            drive_graph_raw=generator.drive_graph_raw,
            drive_graph_proj=generator.drive_graph_proj,
            seed=random_seed,
            batch_size=5,
        )
        standalone_fitness_cache[key] = cached
    return cached

def get_systemic_fitness(route, *, use_full: bool = False):
    key = route_signature(route)
    cache = systemic_fitness_cache
    cached = cache.get((key, use_full))
    if cached is None:
        evaluator = full_systemic_evaluator if use_full else note_systemic_evaluator
        cached = evaluator.evaluate(route, seed=random_seed)
        cache[(key, use_full)] = cached
    return cached

def build_route_notes(summary_df: pd.DataFrame, routes: list) -> dict[str, dict[str, str]]:
    notes: dict[str, dict[str, str]] = {}
    route_by_id = {r.route_id: r for r in routes}
    total = len(summary_df)
    if total == 0:
        return notes
    
    for row in tqdm(summary_df.itertuples(index=False), total=total, desc="Building route notes"):
        route = route_by_id.get(row.route_id)
        interpretation = interpret_embeddings(route) if route else "Route data unavailable."
        if route is not None:
            standalone = get_standalone_fitness(route)
            systemic = get_systemic_fitness(route)
            interpretation = (
                f"{format_route_fitness(standalone)}\n"
                f"{format_systemic_fitness(systemic)}\n\n"
                f"{interpretation}"
            )
        notes[row.route_id] = {
            "encoding": build_route_encoding(row.route_id),
            "interpretation": interpretation,
        }
    return notes

route_notes = build_route_notes(summary, routes)


Building route notes: 100%|██████████| 20/20 [09:43<00:00, 29.20s/it]


## 3. 3-Layer Graph Reward Formulation
Connect the generated physical shape to the 3-layer behavioral passenger graph to calculate the fitness score (Reward).

In [4]:
comparison_generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=1,
)
comparison_routes = comparison_generator.generate_routes(2, route_prefix='CMP', seed=1)
good_shape = comparison_routes[0].path_node_ids
bad_shape = comparison_routes[1].path_node_ids

good_result = calculate_route_fitness(
    good_shape,
    passenger_map=comparison_generator.passenger_map,
    drive_graph_raw=comparison_generator.drive_graph_raw,
    drive_graph_proj=comparison_generator.drive_graph_proj,
    seed=1,
    batch_size=5,
)
bad_result = calculate_route_fitness(
    bad_shape,
    passenger_map=comparison_generator.passenger_map,
    drive_graph_raw=comparison_generator.drive_graph_raw,
    drive_graph_proj=comparison_generator.drive_graph_proj,
    seed=1,
    batch_size=5,
)

comparison = pd.DataFrame(
    [
        {"route_shape": "good", "reward": round(good_result.reward, 3), "avg_gtc": round(good_result.average_gtc, 3), "unserved": good_result.unserved_passenger_count},
        {"route_shape": "bad", "reward": round(bad_result.reward, 3), "avg_gtc": round(bad_result.average_gtc, 3), "unserved": bad_result.unserved_passenger_count},
    ]
)
print(f"Good route reward: {good_result.reward:.3f}")
print(f"Bad route reward: {bad_result.reward:.3f}")
comparison


Good route reward: -174.980
Bad route reward: -176.625


,route_shape,reward,avg_gtc,unserved
0,good,-174.980,174.980,0
1,bad,-176.625,176.625,0


## 4. Systemic Synergy Evaluation Utility
Build a statistical evaluation utility that tests a candidate route across multiple background networks to measure true system synergy.

In [5]:
systemic_result = full_systemic_evaluator.evaluate(routes[0], seed=random_seed)
print(f"Systemic average GTC: {systemic_result.average_gtc:.3f}")
print(f"Systemic GTC std dev: {systemic_result.std_gtc:.3f}")
print(f"Passenger GTC std avg: {systemic_result.average_passenger_gtc_std:.3f}")
print(f"Tests used: {systemic_result.n_tests}")


Systemic average GTC: 5925.934
Systemic GTC std dev: 1271.647
Passenger GTC std avg: 14782.615
Tests used: 20


## 5. PPO Training Loop

Trains a PPO policy on the coordinate-invariant route environment and saves the best/worst completed routes it discovers.

In [ ]:
TRAIN_OUTPUT_DIR = OUTPUT_DIR / "ppo_training"
training_artifacts = train_route_agent(
    passenger_map=generator.passenger_map,
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    output_dir=TRAIN_OUTPUT_DIR,
    seed=random_seed,
    total_timesteps=3000,
    systemic_test_mean=2,
    systemic_test_std=0,
    background_route_mean=1,
    background_route_std=0,
    systemic_batch_size=8,
    systemic_std_penalty_weight=1.0,  # Deep RL for Transit Network Design: https://arxiv.org/abs/2502.17758
    ppo_kwargs={
        "n_steps": 256,
        "batch_size": 64,
        "learning_rate": 3e-4,
        "gamma": 0.99,
        "ent_coef": 0.01,
        "verbose": 1,
    },
    env_kwargs={
        "max_steps": 48,
        "min_route_nodes": 6,
    },
)
print(f"Training complete. Best episode return: {training_artifacts.best_snapshot.episode_return:.3f}" if training_artifacts.best_snapshot else "Training complete. No closed-loop route was captured.")
print(f"Worst episode return: {training_artifacts.worst_snapshot.episode_return:.3f}" if training_artifacts.worst_snapshot else "No worst-route snapshot was captured.")

if training_artifacts.best_snapshot is not None:
    display(IFrame(training_artifacts.best_snapshot.output_html.as_uri(), width=1200, height=900))
if training_artifacts.worst_snapshot is not None:
    display(IFrame(training_artifacts.worst_snapshot.output_html.as_uri(), width=1200, height=900))


## For Evaluation: Export HTML Tester

In [6]:
html_path = route_system.export_route_toggle_html(
    route_manager,
    OUTPUT_DIR / "B4_route_explorer.html",
    title=f"B4 Jeepney Routes ({NUM_ROUTES})",
    route_notes=route_notes,
)
print(html_path)
display(IFrame(html_path.as_uri(), width=1200, height=900))


C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\results\B4_baseline_routes\B4_route_explorer.html
